#   Download resource and model


In [4]:
# ! gdown 1ztoM-hEEnTYUmwz2msCDwmQxfOA7Dupt -O 2_move_student.mp4
# ! gdown 1LP4SnR0R8d-uwPVIPgFL3LWqLa90f3zM -O 4_Move_studet.mp4
# ! gdown 1BNpBRifbDR5rP4NqnL52qPMXUDgPnUn5 -O 6_Move_student.mp4
# ! gdown 10MTr2dNMZlqcuLXgW6VuOhPcP8D82E20 -O 8_Move_student.mp4
# ! gdown 14JKmUAXm-lduq2GL_e8pEL5IhtoYQ9j1 -O 2_Move_rotate_student.mp4
! gdown 1Yl1Eyt7UKEcODOK-_iOkA2x8AXp91vOo -O best.pt
! gdown 1kYbLi1sxY4ZDNbqAkbI8YFm2ZyfZ98Sb -O board.pt


Downloading...
From: https://drive.google.com/uc?id=1Yl1Eyt7UKEcODOK-_iOkA2x8AXp91vOo
To: /content/best.pt
100% 22.5M/22.5M [00:00<00:00, 137MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1kYbLi1sxY4ZDNbqAkbI8YFm2ZyfZ98Sb
From (redirected): https://drive.google.com/uc?id=1kYbLi1sxY4ZDNbqAkbI8YFm2ZyfZ98Sb&confirm=t&uuid=126c0f22-bc7e-4c92-baa9-c59b6a45975d
To: /content/board.pt
100% 124M/124M [00:01<00:00, 87.9MB/s]


# Install necessary library

In [3]:
!pip uninstall ultralytics -y
!pip install ultralytics --upgrade
!pip install torch torchvision --upgrade
!pip install chess
!pip install paddlepaddle paddleocr paddlepaddle-gpu
!pip install --upgrade langchain
!pip install langchain==0.3.27

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/

# Import necessary library

In [5]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
import time
import torch
import torch.nn as nn
from paddleocr import PaddleOCR
import bisect
from types import WrapperDescriptorType

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [6]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# Class BoardSegmentation

In [7]:
class ChessBoardSegmentationModel(nn.Module):
    """U-Net based segmentation model for detecting chess board"""

    class ConvBlock(nn.Module):
        def __init__(self, in_c, out_c):
            super().__init__()
            self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1)
            self.bn1 = nn.BatchNorm2d(out_c)
            self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1)
            self.bn2 = nn.BatchNorm2d(out_c)
            self.relu = nn.ReLU()

        def forward(self, inputs):
            x = self.conv1(inputs)
            x = self.bn1(x)
            x = self.relu(x)
            x = self.conv2(x)
            x = self.bn2(x)
            x = self.relu(x)
            return x

    class EncoderBlock(nn.Module):
        def __init__(self, in_c, out_c):
            super().__init__()
            self.conv = ChessBoardSegmentationModel.ConvBlock(in_c, out_c)
            self.pool = nn.MaxPool2d((2, 2))

        def forward(self, inputs):
            x = self.conv(inputs)
            p = self.pool(x)
            return x, p

    class DecoderBlock(nn.Module):
        def __init__(self, in_c, out_c):
            super().__init__()
            self.up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2, padding=0)
            self.conv = ChessBoardSegmentationModel.ConvBlock(out_c+out_c, out_c)

        def forward(self, inputs, skip):
            x = self.up(inputs)
            x = torch.cat([x, skip], axis=1)
            x = self.conv(x)
            return x

    class BuildUNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.e1 = ChessBoardSegmentationModel.EncoderBlock(3, 64)
            self.e2 = ChessBoardSegmentationModel.EncoderBlock(64, 128)
            self.e3 = ChessBoardSegmentationModel.EncoderBlock(128, 256)
            self.e4 = ChessBoardSegmentationModel.EncoderBlock(256, 512)
            self.b = ChessBoardSegmentationModel.ConvBlock(512, 1024)
            self.d1 = ChessBoardSegmentationModel.DecoderBlock(1024, 512)
            self.d2 = ChessBoardSegmentationModel.DecoderBlock(512, 256)
            self.d3 = ChessBoardSegmentationModel.DecoderBlock(256, 128)
            self.d4 = ChessBoardSegmentationModel.DecoderBlock(128, 64)
            self.outputs = nn.Conv2d(64, 1, kernel_size=1, padding=0)

        def forward(self, inputs):
            s1, p1 = self.e1(inputs)
            s2, p2 = self.e2(p1)
            s3, p3 = self.e3(p2)
            s4, p4 = self.e4(p3)
            b = self.b(p4)
            d1 = self.d1(b, s4)
            d2 = self.d2(d1, s3)
            d3 = self.d3(d2, s2)
            d4 = self.d4(d3, s1)
            outputs = self.outputs(d4)
            return outputs

    def __init__(self):
        super(ChessBoardSegmentationModel, self).__init__()
        self.arc = self.BuildUNet()

    def forward(self, images, masks=None):
        logits = self.arc(images)
        return logits



# Class ChessMovementDetector

In [19]:
class ChessMovementDetector:
    """Main class for detecting chess movements from video"""

    PIECE_MAPPING = {
        "black-pawn": 1,
        "black-bishop": 2,
        "black-knight": 3,
        "black-rook": 4,
        "black-queen": 5,
        "black-king": 6,
        "white-pawn": 11,
        "white-bishop": 12,
        "white-knight": 13,
        "white-rook": 14,
        "white-queen": 15,
        "white-king": 16,
        "empty": 0,
        16: "white-king",
        15: "white-queen",
        14: "white-rook",
        13: "white-knight",
        12: "white-bishop",
        11: "white-pawn",
        6: "black-king",
        5: "black-queen",
        4: "black-rook",
        3: "black-knight",
        2: "black-bishop",
        1: "black-pawn"
    }

    COL_MAPPING = {0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', -1: 'X'}

    CHESS_NOTATION = {
        'pawn': '',
        'knight': 'N',
        'bishop': 'B',
        'rook': 'R',
        'queen': 'Q',
        'king': 'K'
    }

    def __init__(self, piece_detection_model_path, board_segmentation_model_path,
                 frame_skip=15, movement_threshold=140, stability_threshold=5):
        """
        Initialize the chess movement detector

        Args:
            piece_detection_model_path: Path to YOLO model for piece detection
            board_segmentation_model_path: Path to U-Net model for board segmentation
            frame_skip: Process every Nth frame (default: 15)
            movement_threshold: Threshold for movement detection (default: 140)
            stability_threshold: Frames needed to confirm stability (default: 5)
        """
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

        # Load models
        self.piece_model = YOLO(piece_detection_model_path)
        self.board_model = self._load_board_model(board_segmentation_model_path)
        self.ocr = PaddleOCR(use_textline_orientation=True, lang='en')

        # Parameters
        self.frame_skip = frame_skip
        self.movement_threshold = movement_threshold
        self.stability_threshold = stability_threshold

        # State variables
        self.frames = []
        self.board_states = []
        self.board_matrix = []
        self.board_locate = []
        self.board_conf = []
        self.transform_matrix = None
        self.x_limits = []
        self.y_limits = []
        self.warped_img = None
        self.rotation = "normal"

    def _load_board_model(self, model_path):
        """Load the board segmentation model"""
        model = ChessBoardSegmentationModel()
        model.load_state_dict(torch.load(model_path))
        model.eval()
        model.to(self.device)
        return model

    def load_video(self, video_path):
        """Load and extract frames from video"""
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            raise ValueError(f"Error: Could not open video at {video_path}")

        self.frames = []
        frame_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            if frame_count % self.frame_skip == 0:
                self.frames.append(frame)

        cap.release()
        # print(f"Loaded {len(self.frames)} frames from video")
        return len(self.frames)

    def _predict_on_large_image(self, image, image_size=(224, 224), overlap_ratio=0.25):
        """Predict segmentation on large image using sliding window"""
        large_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        original_h, original_w, _ = large_image.shape

        patch_h, patch_w = image_size
        stride_h = int(patch_h * (1 - overlap_ratio))
        stride_w = int(patch_w * (1 - overlap_ratio))

        full_prediction = np.zeros((original_h, original_w), dtype=np.float32)
        overlap_counter = np.zeros((original_h, original_w), dtype=np.float32)

        with torch.no_grad():
            for y in range(0, original_h, stride_h):
                for x in range(0, original_w, stride_w):
                    y_end = min(y + patch_h, original_h)
                    x_end = min(x + patch_w, original_w)
                    y_start = max(0, y_end - patch_h)
                    x_start = max(0, x_end - patch_w)

                    patch = large_image[y_start:y_end, x_start:x_end]
                    processed_patch = cv2.resize(patch, (image_size[1], image_size[0]))
                    processed_patch = np.transpose(processed_patch, (2, 0, 1)).astype(np.float32)
                    processed_patch = torch.Tensor(processed_patch) / 255.0
                    processed_patch = processed_patch.unsqueeze(0).to(self.device)

                    output_logits = self.board_model(processed_patch)
                    output_mask = torch.sigmoid(output_logits).squeeze().cpu().numpy()
                    output_mask_resized = cv2.resize(output_mask, (x_end - x_start, y_end - y_start))

                    full_prediction[y_start:y_end, x_start:x_end] += output_mask_resized
                    overlap_counter[y_start:y_end, x_start:x_end] += 1

        full_prediction /= overlap_counter
        final_mask = (full_prediction > 0.5).astype(np.uint8) * 255

        return large_image, final_mask

    def detect_board(self, frame_index=0, width=1000, height=1000, threshold=60):
        """Detect and warp the chess board from a frame"""
        if not self.frames:
            raise ValueError("No frames loaded. Call load_video() first.")

        original_image, segmentation_mask = self._predict_on_large_image(self.frames[frame_index])

        # Morphological closing
        kernel = np.ones((7, 7), np.uint8)
        closed_image = cv2.morphologyEx(segmentation_mask, cv2.MORPH_CLOSE, kernel, iterations=1)

        # Find contours and get largest
        contours, _ = cv2.findContours(closed_image, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        largest_contour = max(contours, key=cv2.contourArea)

        # Find corner points
        top_left = top_right = bottom_left = bottom_right = None

        for point in largest_contour[:, 0]:
            x, y = point
            if top_left is None or (x + y < top_left[0] + top_left[1]):
                top_left = (x, y)
            if top_right is None or (x - y > top_right[0] - top_right[1]):
                top_right = (x, y)
            if bottom_left is None or (x - y < bottom_left[0] - bottom_left[1]):
                bottom_left = (x, y)
            if bottom_right is None or (x + y > bottom_right[0] + bottom_right[1]):
                bottom_right = (x, y)

        # Perspective transform
        extreme_points = np.float32([top_left, top_right, bottom_left, bottom_right])
        dst_pts = np.float32([
            [threshold, threshold],
            [width + threshold, threshold],
            [threshold, height + threshold],
            [width + threshold, height + threshold]
        ])

        self.transform_matrix = cv2.getPerspectiveTransform(extreme_points, dst_pts)
        warped_img = cv2.warpPerspective(original_image, self.transform_matrix,
                                        (width + 2 * threshold, height + 2 * threshold))

        self.warped_img = warped_img
        return warped_img
    def proprocess_rotate(self):
        rgb = cv2.cvtColor(self.warped_img[900:1060,60:1060], cv2.COLOR_BGR2RGB)
        result = self.ocr.predict(rgb)

        if result and result[0] and 'rec_texts' in result[0] and 'rec_polys' in result[0]:
            recognized_texts = result[0]['rec_texts']
            recognized_polys = result[0]['rec_polys']

            collected_character_positions = []
            for i, (text, poly) in enumerate(zip(recognized_texts, recognized_polys)):
              char_info = {
                'character': text,
                'position': poly.tolist()
              }
              collected_character_positions.append(char_info)
            image_with_boxes = self.warped_img.copy()
            for i, (text, poly) in enumerate(zip(recognized_texts, recognized_polys)):
                pts = poly.astype(int).reshape((-1, 1, 2))
                cv2.polylines(image_with_boxes, [pts], True, (0, 255, 0), 2) # Green color, thickness 2
                cv2.putText(image_with_boxes, text, (pts[0][0][0], pts[0][0][1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        else :
          # print("no detection")
          return 0

        alphabets = "abcdefgh"
        numbers = "12345678"

        isAlphabet = []
        isNumber = []

        for i in range(len(collected_character_positions)):
          if collected_character_positions[i]["character"] in alphabets:
            isAlphabet.append(collected_character_positions[i]["character"].lower())
          elif collected_character_positions[i]["character"] in numbers:
            isNumber.append(int(collected_character_positions[i]["character"]))

        # print("Alpha", isAlphabet)
        # print("Number", isNumber)

        if len(isAlphabet) > len(isNumber):
          if len(isAlphabet) >= 2:
            if isAlphabet[0] > isAlphabet[-1]:
              self.rotation = "flipside"
              return "flipside"
        else:
          if len(isNumber) >= 2:
            if isNumber[0] > isNumber[-1]:
              self.rotation = "clockwise"
              return "clockwise"
          self.rotation = "counterclockwise"
          return "counterclockwise"

        self.rotation = "normal"
        return "normal"

    def detect_grid_lines(self, warped_img):
        """Detect grid lines on the warped board"""
        gray_img = cv2.cvtColor(warped_img[60:1065, 65:1060], cv2.COLOR_RGB2GRAY)

        # Gamma correction
        gamma = 2
        normalized_img = gray_img.astype(np.float32) / 255.0
        corrected_img = np.power(normalized_img, gamma)
        bias_img = (corrected_img * 255).astype(np.uint8)

        # Canny edge detection
        img_blur = cv2.GaussianBlur(bias_img, (5, 5), 0)
        median = np.median(bias_img)
        edge_img = cv2.Canny(img_blur, median * 0.8, median * 1.2)
        upper = int(median * 1.2)

        # Dilate edges
        kernel = np.ones((7, 7), np.uint8)
        dilated_img = cv2.dilate(edge_img, kernel, iterations=1)

        # Hough line detection
        lines = cv2.HoughLinesP(dilated_img, 1, np.pi/180, threshold=upper,
                               minLineLength=700, maxLineGap=50)

        # Get grid limits
        self.x_limits, self.y_limits = self._get_grid_limits(lines)

        return lines

    def _get_grid_limits(self, all_lines, threshold=40):
        """Extract and merge grid line coordinates"""
        raw_x = []
        raw_y = []

        for line in all_lines:
            x1, y1, x2, y2 = line[0]
            if abs(x2 - x1) > abs(y2 - y1):
                raw_y.append((y1 + y2) // 2)
            else:
                raw_x.append((x1 + x2) // 2)

        def merge_coordinates(coords):
            coords.sort()
            if not coords:
                return []

            merged = []
            current_group = [coords[0]]

            for val in coords[1:]:
                if val - current_group[-1] < threshold:
                    current_group.append(val)
                else:
                    merged.append(int(np.mean(current_group)))
                    current_group = [val]

            merged.append(int(np.mean(current_group)))
            return merged

        return merge_coordinates(raw_x), merge_coordinates(raw_y)

    def _is_different(self, image1, image2, lim):
        """Check if two images are different"""
        diff = cv2.absdiff(image1, image2)
        return np.max(diff) > lim

    def detect_movements(self):
        """Detect all chess piece movements in the video"""
        if self.transform_matrix is None:
            raise ValueError("Board not detected. Call detect_board() first.")

        self.board_states = []
        last_processed_stable_board_state = None
        consecutive_static_frames = 0
        in_movement_phase = False

        width, height, threshold = 1000, 1000, 60

        # Process initial frame
        if len(self.frames) > 0:
            image_resized_initial = cv2.warpPerspective(
                self.frames[0], self.transform_matrix,
                (width + 2 * threshold, height + 2 * threshold)
            )

            results_initial = self.piece_model(image_resized_initial)
            initial_board_state_dict = self._process_detection_results(results_initial)

            self.board_states.append(initial_board_state_dict)
            last_processed_stable_board_state = initial_board_state_dict

        # Process subsequent frames
        for i in range(1, len(self.frames)):
            movement_detected = self._is_different(
                self.frames[i-1], self.frames[i], self.movement_threshold
            )

            if movement_detected:
                in_movement_phase = True
                consecutive_static_frames = 0
            else:
                consecutive_static_frames += 1

                if in_movement_phase and consecutive_static_frames >= self.stability_threshold:
                    image_resized = cv2.warpPerspective(
                        self.frames[i], self.transform_matrix,
                        (width + 2 * threshold, height + 2 * threshold)
                    )

                    results = self.piece_model(image_resized)
                    current_board_state_dict = self._process_detection_results(results)

                    if current_board_state_dict != last_processed_stable_board_state:
                        self.board_states.append(current_board_state_dict)
                        last_processed_stable_board_state = current_board_state_dict

                    in_movement_phase = False
                    consecutive_static_frames = 0

        # print(f"Detected {len(self.board_states)} unique board states")
        return len(self.board_states)

    def _process_detection_results(self, results):
        """Process YOLO detection results into board state dictionary"""
        board_state_dict = {}

        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            class_name = results[0].names[cls]

            if class_name not in board_state_dict:
                board_state_dict[class_name] = []

            # Calculate validated coordinates
            y_validate = int(y2 + y1) // 2 - 60
            x_validate = int(x2 + x1) // 2 - 60

            # Apply position adjustments
            if y1 <= 60:
                y_validate += 30
            elif y2 >= 1060:
                y_validate -= 30
            elif y2 <= 530:
                y_validate += 10
            else:
                y_validate -= 10

            if x1 <= 60:
                x_validate += 30
            elif x2 >= 1060:
                x_validate -= 30
            elif x2 <= 530:
                x_validate += 10
            else:
                x_validate -= 10

            board_state_dict[class_name].append((x_validate, y_validate, conf))

        return board_state_dict

    def create_board_matrices(self):
        """Convert board states to matrices"""
        self.board_matrix = []
        self.board_locate = []
        self.board_conf = []

        for state in self.board_states:
            matrix = np.zeros((8, 8))
            location = np.zeros((8, 8))
            confi = np.zeros((8, 8))

            # Collect all pieces with confidence
            all_pieces = []
            for piece_type, positions in state.items():
                for x_coord, y_coord, conf_score in positions:
                    all_pieces.append({
                        'type': piece_type,
                        'x': x_coord,
                        'y': y_coord,
                        'conf': conf_score
                    })

            # Sort by confidence (descending)
            all_pieces.sort(key=lambda p: p['conf'], reverse=True)

            # Place pieces on board (highest confidence first)
            for piece in all_pieces:
                col = bisect.bisect_right(self.x_limits, piece['x']) - 1
                row = bisect.bisect_right(self.y_limits, piece['y']) - 1

                if 0 <= row < 8 and 0 <= col < 8 and matrix[row][col] == 0:
                    matrix[row][col] = self.PIECE_MAPPING[piece['type']]
                    location[row][col] = 1
                    confi[row][col] = piece['conf']

            # Only add if different from last state
            new_matrix = matrix.copy()
            new_location = location.copy()
            new_confi = confi.copy()
            # print(self.rotation)

            if (self.rotation == "counterclockwise"):
                new_matrix = np.rot90(matrix)
                new_location = np.rot90(location)
                new_confi = np.rot90(confi)
            elif (self.rotation == "clockwise"):
                new_matrix = np.rot90(matrix, 3)
                new_location = np.rot90(location, 3)
                new_confi = np.rot90(confi, 3)
            elif (self.rotation == "normal"):
                pass
            elif (self.rotation == "flipside"):
                new_matrix = np.rot90(matrix,2)
                new_location = np.rot90(location,2)
                new_confi = np.rot90(confi,2)

            if len(self.board_matrix) == 0 or not np.array_equal(self.board_matrix[-1], matrix):
                self.board_matrix.append(new_matrix)
                self.board_locate.append(new_location)
                self.board_conf.append(new_confi)

        # print(f"Created {len(self.board_matrix)} board matrices")

    def generate_pgn(self):
        """Generate PGN notation from detected moves"""
        # Validate matrices

        for i in range(len(self.board_matrix) - 1):
            self._validate_matrix(i, i + 1)
        for i in range(len(self.board_matrix) - 1, 0, -1):
            self._validate_matrix(i, i - 1)

        pgn = ""
        round_num = 1
        turn = 'white'
        wf = False

        for index in range(len(self.board_matrix) - 1):
            diff_locate = np.array(self.board_locate[index + 1]) - np.array(self.board_locate[index])
            if np.all(diff_locate == 0):
               continue

            if turn == 'white':
                pgn += str(round_num) + '. '
            if np.sum(diff_locate) >= 1 :
                pgn += "KKK"
                if turn == 'black':
                   round_num += 1
                   turn = 'white'
                else:
                   turn = 'black'
                continue
            # print(pgn)
            # print(f'this matrix state {index} compare to the {index + 1}')
            # np.set_printoptions(threshold=np.inf)
            # print(diff_locate)

            # Check if this move is a capture
            if np.count_nonzero(diff_locate) == 1:
                # exactly one value is non-zero:
                i, j = self._find_capture(index, index + 1)
                piece_id = self.board_matrix[index + 1][i][j]

                if not wf and round_num == 1 and 'black' in self.PIECE_MAPPING[piece_id]:
                    pgn += ".. "
                    turn = 'black'
                else : wf = True

                # print("Capture Move")
                # print(piece_id)
                if piece_id in (-1,0):
                   pgn += "KKK "
                   if turn == 'black':
                      round_num += 1
                      turn = 'white'
                   else:
                      turn = 'black'
                   continue
                piece_type = self.PIECE_MAPPING[piece_id].split('-')[1]
                notation = self.CHESS_NOTATION[piece_type]

                # For pawn captures, include the source file
                if piece_type == 'pawn':
                    # Find the source column
                    for src_i in range(8):
                        for src_j in range(8):
                            if diff_locate[src_i][src_j] < 0:
                                notation = self.COL_MAPPING[src_j]
                                break

                pgn += notation + 'x' + self.COL_MAPPING[j] + str(8 - i)

                if self._is_check(i, j, piece_id, index + 1):
                    pgn += '+'
            else:
                # Regular move (not a capture)
                row, col, piece_id = -1, -1, -1
                for i in range(8):
                    for j in range(8):
                        if diff_locate[i][j] > 0:
                            piece_id = self.board_matrix[index + 1][i][j]
                            row, col = i, j
                            break

                if not wf and round_num == 1 and piece_id > 0 and 'black' in self.PIECE_MAPPING[piece_id]:
                    pgn += ".. "
                    turn = 'black'
                else: wf = True

                if piece_id in (-1,0):
                   pgn += "KKK "
                   if turn == 'black':
                      round_num += 1
                      turn = 'white'
                   else:
                      turn = 'black'
                   continue

                # print("Regular Move")
                # print(piece_id)

                try:
                    piece_type = self.PIECE_MAPPING[piece_id].split('-')[1]
                    pgn += self.CHESS_NOTATION[piece_type] + self.COL_MAPPING[col] + str(8 - row)

                    if self._is_check(row, col, piece_id, index + 1):
                      pgn += '+'
                except (ValueError, TypeError) as e:
                      break

            if turn == 'black':
                round_num += 1
                turn = 'white'
            else:
                turn = 'black'

            if index < len(self.board_matrix) - 2:
                pgn += " "

        return pgn

    def _validate_matrix(self, index1, index2):
        """Validate and correct board matrices"""
        if index2 < 0 or index2 >= len(self.board_locate):
            return

        diff_locate = np.array(self.board_locate[index2]) - np.array(self.board_locate[index1])

        # Handle each square independently
        for i in range(8):
            for j in range(8):
                # If square is unchanged (both states have same occupancy)
                if diff_locate[i][j] == 0:
                    # If both have pieces, keep the higher confidence one
                    if self.board_matrix[index1][i][j] != 0 and self.board_matrix[index2][i][j] != 0 and self.PIECE_MAPPING[self.board_matrix[index1][i][j]].split('-')[0] == self.PIECE_MAPPING[self.board_matrix[index2][i][j]].split('-')[0] :
                        if self.board_conf[index1][i][j] > self.board_conf[index2][i][j]:
                            self.board_matrix[index2][i][j] = self.board_matrix[index1][i][j]
                            self.board_conf[index2][i][j] = self.board_conf[index1][i][j]
                        else:
                            self.board_matrix[index1][i][j] = self.board_matrix[index2][i][j]
                            self.board_conf[index1][i][j] = self.board_conf[index2][i][j]
                elif diff_locate[i][j] == -1 and self.board_matrix[index1][i][j] !=0:
                     self.board_matrix[index2][i][j] = self.board_matrix[index1][i][j]
                     self.board_conf[index2][i][j] = self.board_conf[index1][i][j]
                     diff_locate[i][j] = 0

        # For moved pieces, validate the move makes sense
        # Find source and destination of the move
        moved_pieces = []
        for i in range(8):
            for j in range(8):
                if diff_locate[i][j] < 0:  # Piece disappeared
                    moved_pieces.append({
                        'type': 'source',
                        'row': i,
                        'col': j,
                        'piece': self.board_matrix[index1][i][j],
                        'conf': self.board_conf[index1][i][j]
                    })
                elif diff_locate[i][j] > 0:  # Piece appeared
                    moved_pieces.append({
                        'type': 'dest',
                        'row': i,
                        'col': j,
                        'piece': self.board_matrix[index2][i][j],
                        'conf': self.board_conf[index2][i][j]
                    })

        # If we have exactly one source and one dest, validate consistency
        sources = [p for p in moved_pieces if p['type'] == 'source']
        dests = [p for p in moved_pieces if p['type'] == 'dest']

        if len(sources) == 1 and len(dests) == 1:
            src = sources[0]
            dst = dests[0]

            # The piece that moved should be the same type (unless capture)
            # Use higher confidence piece identity
            if src['conf'] > dst['conf']:
                self.board_matrix[index2][dst['row']][dst['col']] = src['piece']
                self.board_conf[index2][dst['row']][dst['col']] = src['conf']
            else:
                self.board_matrix[index1][src['row']][src['col']] = dst['piece']
                self.board_conf[index1][src['row']][src['col']] = dst['conf']

    def _is_capture(self, index1, index2):
        """
        Check if move is a capture by examining both board states

        A capture occurs when a square contains opponent pieces before and after
        """
        for i in range(8):
            for j in range(8):
                # Square has pieces both before and after the move
                if (self.board_matrix[index2][i][j] != 0 and
                    self.board_matrix[index1][i][j] != 0):

                    piece_before = self.PIECE_MAPPING[self.board_matrix[index1][i][j]]
                    piece_after = self.PIECE_MAPPING[self.board_matrix[index2][i][j]]

                    # Different colors = capture occurred
                    if piece_before.split('-')[0] != piece_after.split('-')[0]:
                        return True
        return False

    def _find_capture(self, index1, index2):
        """
        Find the position where capture occurred (destination square)

        A capture square is where:
        - There was an opponent's piece before
        - Now there's a different color piece
        """
        for i in range(8):
            for j in range(8):
                # Square has pieces both before and after the move
                if (self.board_matrix[index2][i][j] != 0 and
                    self.board_matrix[index1][i][j] != 0):

                    piece_before = self.PIECE_MAPPING[self.board_matrix[index1][i][j]]
                    piece_after = self.PIECE_MAPPING[self.board_matrix[index2][i][j]]

                    # Different colors = capture occurred here
                    if piece_before.split('-')[0] != piece_after.split('-')[0]:
                        return i, j
        return -1, -1

    def _is_check(self, i, j, piece_id, index):
        """Check if a move puts the opponent in check"""
        move = 1 if 'white' in self.PIECE_MAPPING[piece_id] else -1
        color = 'black' if 'white' in self.PIECE_MAPPING[piece_id] else 'white'

        king_row = king_col = -1
        king_piece = f'{color}-king'

        for k in range(8):
            for l in range(8):
                if (self.board_matrix[index][k][l] != 0 and
                    self.PIECE_MAPPING[self.board_matrix[index][k][l]] == king_piece):
                    king_row, king_col = k, l

        if king_row == -1 or king_col == -1:
            return False

        piece_type = self.PIECE_MAPPING[piece_id].split('-')[1]

        if piece_type == 'pawn' and king_row == i + move and abs(king_col - j) == 1:
            return True
        elif piece_type == 'knight' and (king_row, king_col) in [
            (i+2, j-1), (i+2, j+1), (i-2, j+1), (i-2, j-1),
            (i+1, j+2), (i-1, j+2), (i+1, j-2), (i-1, j-2)
        ]:
            return True
        elif piece_type == 'bishop' and abs(king_row - i) == abs(king_col - j):
            return True
        elif piece_type == 'rook' and (king_row == i or king_col == j):
            return True
        elif piece_type == 'queen' and (king_row == i or king_col == j or abs(king_row - i) == abs(king_col - j)):
            return True
        elif piece_type == 'king' and abs(king_row - i) <= 1 and abs(king_col - j) <= 1:
            return True

        return False

    def process_video(self, video_path):
        """
        Process entire video and return PGN notation

        Args:
            video_path: Path to the chess game video

        Returns:
            PGN notation string
        """
        # print(f"Loading video: {video_path}")
        self.load_video(video_path)

        # print("Detecting board...")
        warped_board = self.detect_board()

        # print("Detecting grid lines...")
        self.detect_grid_lines(warped_board)

        # print("Detecting movements...")
        self.detect_movements()

        # print("Detecting board alignment...")
        self.proprocess_rotate()

        # print("Creating board matrices...")
        self.create_board_matrices()

        # print("Generating PGN...")
        pgn = self.generate_pgn()

        return pgn

In [20]:
detector = ChessMovementDetector(
        piece_detection_model_path='best.pt',
        board_segmentation_model_path='board.pt',
        frame_skip=15,
        movement_threshold=140,
        stability_threshold=5
    )



Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/en_PP-OCRv5_mobile_rec`.


In [9]:
# # Process video and get PGN
# pgn = detector.process_video('8_Move_student.mp4')
# print(f"Generated PGN: {pgn}")

In [10]:
# import pandas as pd

# video_files = ['2_Move_rotate_student.mp4', '2_move_student.mp4', '4_Move_studet.mp4', '6_Move_student.mp4', '8_Move_student.mp4', '(Bonus)Long_video_student.mp4']
# results = []

# for video_file in video_files:
#     # print(f"Processing {video_file}...")
#     try:
#         pgn = detector.process_video(video_file)
#         results.append({"row_id": video_file, "output": pgn})
#     except Exception as e:
#         print(f"Error processing {video_file}: {e}")
#         results.append({"nameOfVideo": video_file, "PGN": "Error: " + str(e)})

# df = pd.DataFrame(results)
# df.to_csv('video_pgns.csv', index=False)

# # print("CSV file 'video_pgns.csv' generated successfully.")
# # print(df)

In [11]:
# df = df.drop(columns=['nameOfVideo', 'PGN'])
# print(df)

In [12]:
# df.to_csv('video_pgns.csv', index=False)

In [10]:
def chess_move(video_name):
  # print(video_name)
  return detector.process_video(video_name)

# Marking
TAs will use this section to check the PGN during the hidden video score evaluation. You can also use it for testing with the evaluation video set.

In [11]:
!pip install editdistance
!pip install opencv-python

In [12]:
!wget https://drive.google.com/uc?id=170ij2-tosvk8Ium67ScjnwKvYXqGkolN -O chess_solution_final_seen.csv

--2025-12-11 14:09:30--  https://drive.google.com/uc?id=170ij2-tosvk8Ium67ScjnwKvYXqGkolN
Resolving drive.google.com (drive.google.com)... 173.194.202.101, 173.194.202.100, 173.194.202.113, ...
Connecting to drive.google.com (drive.google.com)|173.194.202.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=170ij2-tosvk8Ium67ScjnwKvYXqGkolN [following]
--2025-12-11 14:09:30--  https://drive.usercontent.google.com/download?id=170ij2-tosvk8Ium67ScjnwKvYXqGkolN
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.199.132, 2607:f8b0:400e:c02::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.199.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 965 [application/octet-stream]
Saving to: ‘chess_solution_final_seen.csv’

chess_solution_fina 100%[===================>]     965  --.-KB/s    in 0s      

2025-12-11 14:09:32 (38

In [13]:
import subprocess

def download_file(url, filename=None):
    """
    Downloads a file from the specified URL using wget.

    Parameters:
        url (str): The URL of the file to download.
        output_directory (str, optional): The directory to save the downloaded file.
    """
    # Construct the wget command
    cmd = ['wget', url]
    if filename:
        cmd.extend(['-O', filename])

    try:
        # Execute the wget command
        print(cmd)
        subprocess.run(cmd, check=True)
        print(f"Downloaded {url} successfully.")
    except subprocess.CalledProcessError as e:
        print(f"An error occurred: {e}")

### Download .mp4 file

In [14]:
# Load videos (or you can upload videos here manually)
import gdown

f = open('chess_solution_final_seen.csv','r')
head = f.readline()
for line in f:
  video_name, moves_sol, p,link = line.split(',')
  idx = link.find('/d/')
  idx2 = link.find('/',idx+3)
  print(link[idx+3:idx2])
  url = 'https://drive.google.com/uc?id=' + link[idx+3:idx2]

  #
  if (video_name == "(Bonus)Long_Video_label.mp4"): url = 'https://drive.google.com/uc?id=1rHgKqY8nZpbR_BensPmhROwV_HHdcYr5'
  #

  print(url,video_name)
  gdown.download(url, output=video_name, quiet=False)

1xa_KjGYMUyWsWp4-WCmobPt2OBdle8eh
https://drive.google.com/uc?id=1xa_KjGYMUyWsWp4-WCmobPt2OBdle8eh 2_Move_rotate_student.mp4


Downloading...
From: https://drive.google.com/uc?id=1xa_KjGYMUyWsWp4-WCmobPt2OBdle8eh
To: /content/2_Move_rotate_student.mp4
100%|██████████| 58.8M/58.8M [00:00<00:00, 103MB/s] 


1G3pu__vho0cUv7QifV8t3-5c8D4DMQ0v
https://drive.google.com/uc?id=1G3pu__vho0cUv7QifV8t3-5c8D4DMQ0v 2_move_student.mp4


Downloading...
From: https://drive.google.com/uc?id=1G3pu__vho0cUv7QifV8t3-5c8D4DMQ0v
To: /content/2_move_student.mp4
100%|██████████| 58.0M/58.0M [00:00<00:00, 72.1MB/s]


1xFojdWcBqSxlC_VUar-NQpm-raneuRMM
https://drive.google.com/uc?id=1xFojdWcBqSxlC_VUar-NQpm-raneuRMM 4_Move_studet.mp4


...

Downloading...
From (original): https://drive.google.com/uc?id=1y2cYGHxRQoUx2xgL5JLe5x9G-idvVlFR
From (redirected): https://drive.google.com/uc?id=1y2cYGHxRQoUx2xgL5JLe5x9G-idvVlFR&confirm=t&uuid=52f2d203-5119-46c1-af8f-7577cd9ebf3e
To: /content/6_Move_student.mp4
100%|██████████| 109M/109M [00:00<00:00, 171MB/s] 


1s12dq4e1F7QlBNmdfT_4eIBn-OgevvCY
https://drive.google.com/uc?id=1s12dq4e1F7QlBNmdfT_4eIBn-OgevvCY 8_Move_student.mp4


Downloading...
From (original): https://drive.google.com/uc?id=1s12dq4e1F7QlBNmdfT_4eIBn-OgevvCY
From (redirected): https://drive.google.com/uc?id=1s12dq4e1F7QlBNmdfT_4eIBn-OgevvCY&confirm=t&uuid=5089dfd4-0066-4c37-8a34-f90fb1e143e0
To: /content/8_Move_student.mp4
100%|██████████| 567M/567M [00:02<00:00, 196MB/s]



https://drive.google.com/uc?id=1rHgKqY8nZpbR_BensPmhROwV_HHdcYr5 (Bonus)Long_Video_label.mp4


Downloading...
From (original): https://drive.google.com/uc?id=1rHgKqY8nZpbR_BensPmhROwV_HHdcYr5
From (redirected): https://drive.google.com/uc?id=1rHgKqY8nZpbR_BensPmhROwV_HHdcYr5&confirm=t&uuid=831dfb00-c455-4162-bd12-134530b0cb04
To: /content/(Bonus)Long_Video_label.mp4
100%|██████████| 541M/541M [00:07<00:00, 69.8MB/s]


### Scoring your output

In [ ]:
# Read videos
import editdistance

f = open('chess_solution_final_seen.csv','r')
score = 0
n = 0
for line in f:
  video_name, moves_sol, p,link = line.split(',')
  if video_name == "row_id": continue
  # if video_name == "2_Move_rotate_student.mp4": continue
  # if video_name == "2_move_student.mp4": continue
  # if video_name == "4_Move_studet.mp4": continue
  # if video_name == "6_Move_student.mp4": continue
  # if video_name == "8_Move_student.mp4": continue
  # if video_name == "(Bonus)Long_Video_label.mp4": continue
  moves = chess_move(video_name)
  s = 1 - editdistance.eval(moves,moves_sol)  / max(len(moves),len(moves_sol))
  score += s
  print(f"your result: ",moves)
  print(f"solution   : ",moves_sol )
  print(f"The edit distance between your results and solution is {s}.")
  n += 1
f.close()
if n != 0:
  print("-----------------------")
  print("Total score :",score/n)
  print("-----------------------")



0: 416x416 2 black-bishops, 2 black-kings, 2 black-knights, 4 black-pawns, 1 black-queen, 1 black-rook, 5 white-bishops, 2 white-kings, 3 white-knights, 4 white-pawns, 2 white-queens, 2 white-rooks, 8.4ms
Speed: 2.0ms preprocess, 8.4ms inference, 13.1ms postprocess per image at shape (1, 3, 416, 416)

0: 416x416 2 black-bishops, 3 black-kings, 2 black-knights, 4 black-pawns, 1 black-queen, 1 black-rook, 5 white-bishops, 2 white-kings, 2 white-knights, 5 white-pawns, 2 white-queens, 2 white-rooks, 7.8ms
Speed: 2.3ms preprocess, 7.8ms inference, 15.9ms postprocess per image at shape (1, 3, 416, 416)

0: 416x416 3 black-bishops, 3 black-kings, 3 black-knights, 4 black-pawns, 1 black-rook, 6 white-bishops, 2 white-kings, 3 white-knights, 2 white-pawns, 3 white-queens, 5 white-rooks, 6.7ms
Speed: 2.5ms preprocess, 6.7ms inference, 13.5ms postprocess per image at shape (1, 3, 416, 416)

0: 416x416 2 black-bishops, 3 black-kings, 2 black-knights, 3 black-pawns, 1 black-queen, 1 black-rook, 5